In [1]:
import polars as pl
from collections import defaultdict
import pandas as pd
import os

In [2]:
data_folder = "data"
os.makedirs(data_folder, exist_ok=True)

symbol = "BTCUSDT"
year = 2026
months = list(range(1, 4))
chunksize = 10_000_000 
price_round = 0.5 

In [3]:
for month in months:
    month_str = f"{month:02d}"
    input_file = os.path.join(data_folder, f"{symbol}-aggTrades-{year}-{month_str}.csv")
    output_file = os.path.join(data_folder, f"{symbol}-aggTrades-{year}-{month_str}.parquet")

    if not os.path.exists(input_file):
        print(f"File {input_file} not found, skipping")
        continue

    print(f"Processing {input_file}")

    df_stream = pl.scan_csv(input_file)

    df_processed = (
        df_stream
        .with_columns([
            pl.col("price").cast(pl.Float64),
            pl.col("quantity").cast(pl.Float64),
            pl.from_epoch(pl.col("transact_time"), time_unit="ms").alias("timestamp"),

            ((pl.col("price") / price_round).round(0) * price_round).alias("price_round"),

            pl.when(pl.col("is_buyer_maker"))
            .then(pl.col("quantity"))
            .otherwise(0)
            .alias("ask_qty"),


            pl.when(~pl.col("is_buyer_maker"))
            .then(pl.col("quantity"))
            .otherwise(0)
            .alias("bid_qty")
        ])
        .collect(streaming=True)
    )

    df_processed.write_parquet(output_file, compression="snappy")

    print(f"Saved processed data to {output_file}\n")

Processing data/BTCUSDT-aggTrades-2026-01.csv


/var/folders/1j/l33btpt130g75b5zd4tm00140000gn/T/ipykernel_31262/3535066927.py:34: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


Saved processed data to data/BTCUSDT-aggTrades-2026-01.parquet

Processing data/BTCUSDT-aggTrades-2026-02.csv
Saved processed data to data/BTCUSDT-aggTrades-2026-02.parquet

Processing data/BTCUSDT-aggTrades-2026-03.csv
Saved processed data to data/BTCUSDT-aggTrades-2026-03.parquet



In [4]:
df = pl.read_parquet("data/BTCUSDT-aggTrades-2025-01.parquet")
df.head(20)

agg_trade_id,price,quantity,first_trade_id,last_trade_id,transact_time,is_buyer_maker,timestamp,price_round,ask_qty,bid_qty
i64,f64,f64,i64,i64,i64,bool,datetime[ms],f64,f64,f64
2496124811,93548.8,0.056,5793017800,5793017801,1735689600051,false,2025-01-01 00:00:00.051,93549.0,0.0,0.056
2496124812,93548.7,0.448,5793017802,5793017804,1735689605048,true,2025-01-01 00:00:05.048,93548.5,0.448,0.0
2496124813,93548.8,0.004,5793017805,5793017805,1735689605057,false,2025-01-01 00:00:05.057,93549.0,0.0,0.004
2496124814,93548.7,0.119,5793017806,5793017809,1735689605065,true,2025-01-01 00:00:05.065,93548.5,0.119,0.0
2496124815,93548.8,0.031,5793017810,5793017813,1735689605078,false,2025-01-01 00:00:05.078,93549.0,0.0,0.031
…,…,…,…,…,…,…,…,…,…,…
2496124826,93546.5,0.004,5793017834,5793017834,1735689605124,true,2025-01-01 00:00:05.124,93546.5,0.004,0.0
2496124827,93546.4,0.012,5793017835,5793017835,1735689605124,true,2025-01-01 00:00:05.124,93546.5,0.012,0.0
2496124828,93546.2,0.013,5793017836,5793017836,1735689605124,true,2025-01-01 00:00:05.124,93546.0,0.013,0.0
